In [1]:
# Skipped in CI: Colab/bootstrap dependency install cell.


# SegmentTable: Basics

:::{tip}
For a more comprehensive guide including visualization, GravitySpy integration, and advanced operations, see the [Table / Segment User Guide](intro_table.ipynb).
:::

## GWpy base classes and gwexpy extensions

Segment analysis builds on GWpy types. Each row span is still represented by `gwpy.segments.Segment`, and payload columns can hold GWpy `TimeSeries` or `FrequencySeries` objects directly.

On top of that, gwexpy adds `SegmentTable` as an extension of `gwpy.table.Table`, plus lazy payload handling via `SegmentCell`, and table-oriented batch helpers such as `apply()`, `map()`, `crop()`, and `asd()`. In practice, you keep GWpy base classes for the actual data objects while gwexpy adds the workflow layer for processing many segments together.


In [2]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    import numpy as np
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. Create simple segments
    segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
    st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
    st


## Delayed Loading with SegmentCell

You can add payload columns that are only loaded when needed. This is useful for handling large datasets.

In [3]:
def my_loader():
    # Simulate loading data
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# Add a payload column with a loader (sequence of callables)
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

,span,label,raw
0,"(0, 4)",A,<lazy: timeseries>
1,"(4, 8)",B,<lazy: timeseries>
2,"(8, 12)",C,<lazy: timeseries>


## Row-wise Processing

`SegmentTable` provides an `apply()` method to process each row and collect results into new columns.

In [4]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

,span,label,duration,valid,raw
0,"(0, 4)",A,4.0,True,<lazy: timeseries>
1,"(4, 8)",B,4.0,True,<lazy: timeseries>
2,"(8, 12)",C,4.0,True,<lazy: timeseries>


## Fetch and Conversion

You can explicitly load lazy cells with `fetch()` or `materialize()`. Converting to pandas gives you a standard DataFrame for meta columns.

In [5]:
st2.fetch()
df = st2.to_pandas()
df.head()

Loading series...
Loading series...
Loading series...


,span,label,duration,valid
0,"(0, 4)",A,4.0,True
1,"(4, 8)",B,4.0,True
2,"(8, 12)",C,4.0,True
